# OOP 8: Dataclasses and Pydantic

## Table of Contents

1. [The Problem with Plain Dicts for Configuration](#1)
2. [dataclasses: Structured Data Without Boilerplate](#2)
3. [Pydantic: Validation at the Boundary](#3)
4. [@classmethod and @staticmethod in Practice](#4)
5. [Exercise: Experiment Configuration](#5)

---
## 1. The Problem with Plain Dicts for Configuration <a id="1"></a>

In real ML code, configuration objects get passed everywhere. The naive approach is a plain dict:

```python
config = {
    "model_type": "random_forest",
    "n_estimators": 100,
    "max_depth": None,
    "random_state": 42
}
```

This causes three common bugs:
1. **Typos are silent** — `config["n_estomators"]` raises `KeyError` at the worst possible time
2. **No default values** — every caller must specify every key
3. **No type checking** — `config["n_estimators"] = "100"` is silently wrong

Let's fix these with proper data classes.

---
## 2. dataclasses: Structured Data Without Boilerplate <a id="2"></a>

`@dataclass` auto-generates `__init__`, `__repr__`, and `__eq__` from your type-annotated attributes. It's the standard library's answer to "I just need a struct".

In [ ]:
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class TrainingConfig:
    model_type: str
    n_estimators: int = 100
    max_depth: Optional[int] = None
    random_state: int = 42
    feature_columns: list = field(default_factory=list)


# __init__ is generated automatically
cfg = TrainingConfig(model_type="random_forest", n_estimators=200)
print(cfg)
print()

# __eq__ is generated automatically
cfg2 = TrainingConfig(model_type="random_forest", n_estimators=200)
print("cfg == cfg2:", cfg == cfg2)

# __repr__ is generated automatically
print(repr(cfg))

In [ ]:
# The mutable default trap
# Using a plain list as default would be shared across all instances (Python bug)
# field(default_factory=list) creates a new list for each instance

cfg_a = TrainingConfig(model_type="rf")
cfg_b = TrainingConfig(model_type="rf")

cfg_a.feature_columns.append("tenure")   # only affects cfg_a
print("cfg_a features:", cfg_a.feature_columns)
print("cfg_b features:", cfg_b.feature_columns)  # still empty

In [ ]:
# frozen=True makes the dataclass immutable (like a tuple with names)
from dataclasses import dataclass

@dataclass(frozen=True)
class ModelKey:
    """An immutable identifier for a trained model."""
    model_type: str
    dataset_name: str
    version: int = 1

    def __str__(self) -> str:
        return f"{self.model_type}/{self.dataset_name}/v{self.version}"


key = ModelKey("random_forest", "churn", version=3)
print(key)
print(repr(key))

# Frozen dataclasses can be used as dict keys or in sets
model_registry = {}
model_registry[key] = {"accuracy": 0.91, "auc": 0.87}
print(model_registry[key])

---
## 3. Pydantic: Validation at the Boundary <a id="3"></a>

`dataclasses` don't validate types at runtime — you can still assign `n_estimators = "one hundred"` and it won't complain. **Pydantic** does validate, which makes it ideal for:
- Config files loaded from YAML/JSON
- API request bodies
- Any data entering from the outside world

Install: `pip install pydantic`

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import Optional, Literal


class TrainingConfigPydantic(BaseModel):
    # model_config = ConfigDict(frozen=True)  # replaces class Config in Pydantic v2
    model_type: Literal["random_forest", "gradient_boosting", "logistic_regression"]
    n_estimators: int = Field(100, gt=0, description="Number of trees (must be > 0)")
    max_depth: Optional[int] = Field(None, ge=1, description="Max tree depth (None = unlimited)")
    random_state: int = 42
    feature_columns: list = Field(default_factory=list)

    @field_validator("n_estimators")
    def n_estimators_reasonable(cls, v):
        if v > 10_000:
            raise ValueError(f"n_estimators={v} seems unreasonably large (> 10,000)")
        return v

    class Config:
        frozen = True  # immutable after creation


# Valid config
cfg = TrainingConfigPydantic(model_type="random_forest", n_estimators=200)
print("Valid config:", cfg)
print()

In [ ]:
# Type coercion: Pydantic converts compatible types
cfg_coerced = TrainingConfigPydantic(model_type="random_forest", n_estimators="150")
print("Coerced string to int:", cfg_coerced.n_estimators, type(cfg_coerced.n_estimators))
print()

In [ ]:
# Validation errors: caught at instantiation, not in training
errors_to_test = [
    {"model_type": "xgboost"},                    # invalid literal
    {"model_type": "random_forest", "n_estimators": -5},  # field constraint
    {"model_type": "random_forest", "n_estimators": 50_000},  # custom validator
]

from pydantic import ValidationError

for kwargs in errors_to_test:
    try:
        TrainingConfigPydantic(**kwargs)
    except ValidationError as e:
        print(f"Input: {kwargs}")
        print(f"Error: {e.errors()[0]['msg']}")
        print()

In [ ]:
# Loading config from a dict (e.g. parsed from YAML)
import json

raw_config = """
{
    "model_type": "gradient_boosting",
    "n_estimators": 300,
    "max_depth": 5,
    "feature_columns": ["tenure", "charges", "num_products"]
}
"""

cfg = TrainingConfigPydantic(**json.loads(raw_config))
print("Config loaded from JSON:")
print(cfg)
print()
print("Serialise back to dict:", cfg.dict())

---
## 4. @classmethod and @staticmethod in Practice <a id="4"></a>

Now that we have dataclasses, the most useful application of `@classmethod` and `@staticmethod` is in **factory methods** and **validation helpers**.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json


@dataclass
class ExperimentConfig:
    name: str
    model_type: str
    n_estimators: int = 100
    test_size: float = 0.2
    random_state: int = 42

    # ------------------------------------------------------------------ #
    # Factory methods (classmethod) — alternative constructors            #
    # ------------------------------------------------------------------ #

    @classmethod
    def quick_test(cls, name: str = "quick_test") -> "ExperimentConfig":
        """A config preset for fast iteration during development."""
        return cls(name=name, model_type="logistic_regression", n_estimators=10)

    @classmethod
    def from_json(cls, path: str) -> "ExperimentConfig":
        """Load configuration from a JSON file."""
        with open(path) as f:
            data = json.load(f)
        return cls(**data)

    # ------------------------------------------------------------------ #
    # Static methods — utilities that don't need self or cls              #
    # ------------------------------------------------------------------ #

    @staticmethod
    def valid_model_types() -> list:
        return ["random_forest", "gradient_boosting", "logistic_regression"]

    @staticmethod
    def validate_test_size(test_size: float) -> None:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size must be between 0 and 1, got {test_size}")


# Using the factory methods
cfg_quick = ExperimentConfig.quick_test()
print("Quick test config:", cfg_quick)
print()

cfg_prod = ExperimentConfig(
    name="production_v3",
    model_type="random_forest",
    n_estimators=500,
)
print("Production config:", cfg_prod)
print()

print("Valid model types:", ExperimentConfig.valid_model_types())

try:
    ExperimentConfig.validate_test_size(1.5)
except ValueError as e:
    print(f"Validation error: {e}")

---
## 5. Exercise: Experiment Configuration <a id="5"></a>

Create a `GridSearchConfig` dataclass that represents a hyperparameter search. It should:

1. Have fields: `model_type` (str), `param_grid` (dict), `cv_folds` (int = 5), `scoring` (str = "accuracy"), `n_jobs` (int = -1)
2. Have a `@classmethod from_yaml(path)` factory (you can mock the YAML loading with a dict for now)
3. Have a `@staticmethod n_combinations(param_grid)` that returns how many combinations the grid has
4. Have a method `to_sklearn_params()` that returns the dict needed for `GridSearchCV`

In [ ]:
# Your implementation here
from dataclasses import dataclass, field

@dataclass
class GridSearchConfig:
    pass


# --- Test ---
import numpy as np

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, None],
    "min_samples_split": [2, 5],
}

cfg = GridSearchConfig(
    model_type="random_forest",
    param_grid=param_grid,
    cv_folds=5,
    scoring="roc_auc",
)

print("Config:", cfg)
print("Combinations:", GridSearchConfig.n_combinations(param_grid))  # should be 18
print("sklearn params:", cfg.to_sklearn_params())

> <details>
> <summary>Solution</summary>
>
> ```python
> from dataclasses import dataclass, field
> from math import prod
>
> @dataclass
> class GridSearchConfig:
>     model_type: str
>     param_grid: dict = field(default_factory=dict)
>     cv_folds: int = 5
>     scoring: str = "accuracy"
>     n_jobs: int = -1
>
>     @classmethod
>     def from_yaml(cls, path: str) -> "GridSearchConfig":
>         # In real code: import yaml; data = yaml.safe_load(open(path))
>         data = {"model_type": "random_forest", "cv_folds": 3}
>         return cls(**data)
>
>     @staticmethod
>     def n_combinations(param_grid: dict) -> int:
>         return prod(len(v) for v in param_grid.values())
>
>     def to_sklearn_params(self) -> dict:
>         return {
>             "param_grid": self.param_grid,
>             "cv": self.cv_folds,
>             "scoring": self.scoring,
>             "n_jobs": self.n_jobs,
>         }
> ```
> </details>